In [1]:
pip install torch torchvision matplotlib

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()

        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.01)
        self.bias = nn.Parameter(torch.zeros(out_features))

        self.gate_scores = nn.Parameter(torch.randn(out_features, in_features))

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)   # values between 0 and 1

        pruned_weight = self.weight * gates       # element-wise

        return F.linear(x, pruned_weight, self.bias)

In [3]:
class PrunableNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = PrunableLinear(32*32*3, 512)
        self.fc2 = PrunableLinear(512, 256)
        self.fc3 = PrunableLinear(256, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [4]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor()
])

train_data = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_data = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64)

100%|██████████| 170M/170M [00:13<00:00, 12.4MB/s]


In [5]:
def sparsity_loss(model):
    loss = 0
    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores)
            loss += gates.sum()
    return loss

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = PrunableNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

lambda_val = 0.01  # try different values

for epoch in range(10):
    model.train()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)

        ce_loss = F.cross_entropy(outputs, labels)
        sp_loss = sparsity_loss(model)

        loss = ce_loss + lambda_val * sp_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} Loss: {loss.item()}")

Epoch 0 Loss: 5872.79443359375
Epoch 1 Loss: 3726.533203125
Epoch 2 Loss: 2308.31689453125
Epoch 3 Loss: 1463.52490234375
Epoch 4 Loss: 960.3291625976562
Epoch 5 Loss: 648.062744140625
Epoch 6 Loss: 447.19647216796875
Epoch 7 Loss: 313.0433654785156
Epoch 8 Loss: 221.50245666503906
Epoch 9 Loss: 158.3817901611328


In [7]:
def calculate_sparsity(model):
    total = 0
    zero = 0

    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores)

            total += gates.numel()
            zero += (gates < 1e-2).sum().item()

    return 100 * zero / total

In [8]:
def test(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = outputs.max(1)

            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return 100 * correct / total

In [9]:
lambda_vals = [0.001, 0.01, 0.1]

In [10]:
import matplotlib.pyplot as plt

def plot_gates(model):
    all_gates = []

    for module in model.modules():
        if isinstance(module, PrunableLinear):
            gates = torch.sigmoid(module.gate_scores).detach().cpu().numpy().flatten()
            all_gates.extend(gates)

    plt.hist(all_gates, bins=50)
    plt.title("Gate Distribution")
    plt.show()